# 02. LightGBM baseline reproduction: code-inspection notebook

이 노트북은 CLI 커맨드를 붙여넣는 실행기가 아닙니다. `src/llamac_research/*.py`에 있는 구현을
**기능 단위 코드 셀**로 펼쳐서 사람이 직접 읽고, 셀을 순서대로 실행하며, 바로 다음 셀에서
중간 결과를 확인하도록 구성했습니다.

범위:
1. 원본 노트북 스타일 all-channel feature 생성/검사
2. PPG-only feature 생성/검사
3. LightGBM grouped baseline
4. 원본 노트북에 가까운 official-exclusion + stratified LightGBM 확인

## 0. Notebook runtime configuration

아래 셀은 실험 경로/옵션만 정의합니다. 이후 셀의 함수들은 노트북 안에서 직접 정의됩니다.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import platform
import re
import time
import warnings
from collections.abc import Sequence
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import Markdown, display

def find_repo_root(start: Path | None = None) -> Path:
    """Find repo root from either JupyterLab cwd or notebook-directory cwd."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "llamac_research").exists():
            return candidate
    raise RuntimeError(f"Could not locate llamac_research repo root from {start}")

ROOT = find_repo_root()
DATA_ROOT = ROOT / "data" / "extracted"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "artifacts" / "results"
FEATURE_ALL = PROCESSED / "features_all.parquet"
FEATURE_PPG = PROCESSED / "features_ppg.parquet"

SEED = 42
FOLDS = 5
DEVICE = "cpu"  # portable default; use "auto" only when you want local acceleration.
WORKERS = 4
REBUILD_FEATURES = False
RERUN_EXPERIMENTS = False

RESULTS.mkdir(parents=True, exist_ok=True)
print(ROOT)

## 1. Label contract and official exclusion rules

Source: `src/llamac_research/labels.py`. 이 셀을 실행하면 label mapping, target 생성,
official notebook exclusion 함수가 노트북 namespace에 직접 정의됩니다.

In [ ]:
EMOTION_ID_TO_LABEL: dict[int, str] = {
    1: "neutral",
    2: "fun",
    3: "sadness",
    4: "anger",
    5: "fear",
}
EMOTION_LABELS: list[str] = [EMOTION_ID_TO_LABEL[i] for i in sorted(EMOTION_ID_TO_LABEL)]
EMOTION_IDS: list[int] = sorted(EMOTION_ID_TO_LABEL)

ANSWER_COLUMNS: list[str] = [
    "SubjectID",
    "Trial",
    "NoVideo",
    "Valence",
    "Arousal",
    "Dominance",
    "Liking",
    "EmotType",
    "EmotStr",
    "Seen",
]

TARGET_COLUMNS: list[str] = ["IntendedType", "ReportedType"]
SELF_REPORT_COLUMNS: list[str] = [
    "NoVideo",
    "Valence",
    "Arousal",
    "Dominance",
    "Liking",
    "EmotType",
    "EmotStr",
    "Seen",
    *TARGET_COLUMNS,
]

OFFICIAL_EXCLUDE_SUBJECTS: tuple[int, ...] = (32, 37, 40, 47, 54, 55, 56, 70, 99)
OFFICIAL_EXCLUDE_TRIALS: dict[int, tuple[int, ...]] = {
    7: (27, 28, 36),
    19: (18,),
    59: (35, 13),
    111: (1,),
    20: (26,),
    89: (13,),
    105: (38, 50),
    107: (38,),
}

@dataclass(frozen=True)
class TargetSpec:
    """Resolved target-column metadata."""

    name: str
    column: str
    labels: list[int]
    label_names: list[str]

def map_novideo_to_intended(value: object) -> int | None:
    """Map NoVideo stimulus id to intended emotion id {1..5}."""
    try:
        no_video = int(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return None
    if 1 <= no_video <= 10:
        return 1
    if 11 <= no_video <= 20:
        return 2
    if 21 <= no_video <= 30:
        return 3
    if 31 <= no_video <= 40:
        return 4
    if 41 <= no_video <= 50:
        return 5
    return None

def add_target_columns(frame: pl.DataFrame) -> pl.DataFrame:
    """Add IntendedType and ReportedType columns using the public benchmark mapping."""
    if "NoVideo" not in frame.columns:
        raise ValueError("NoVideo column is required to derive IntendedType")
    if "EmotType" not in frame.columns:
        raise ValueError("EmotType column is required to derive ReportedType")

    intended_expr = (
        pl.when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(1, 10))
        .then(1)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(11, 20))
        .then(2)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(21, 30))
        .then(3)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(31, 40))
        .then(4)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(41, 50))
        .then(5)
        .otherwise(None)
        .cast(pl.Int64)
        .alias("IntendedType")
    )
    return frame.with_columns(
        intended_expr,
        pl.col("EmotType").cast(pl.Int64, strict=False).alias("ReportedType"),
    )

def resolve_target(name: str) -> TargetSpec:
    """Resolve a CLI target name to a dataframe column."""
    normalized = name.lower().strip().replace("-", "_")
    if normalized in {"reported", "reported_type", "self_report", "emottype", "emotion"}:
        return TargetSpec("reported", "ReportedType", EMOTION_IDS, EMOTION_LABELS)
    if normalized in {"intended", "intended_type", "targeted", "novideo"}:
        return TargetSpec("intended", "IntendedType", EMOTION_IDS, EMOTION_LABELS)
    raise ValueError(f"Unsupported target {name!r}; use 'reported' or 'intended'.")

def filter_official_valid_trials(frame: pl.DataFrame) -> pl.DataFrame:
    """Apply the official notebook's subject/trial exclusions."""
    required = {"SubjectID", "Trial"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing columns for official filtering: {sorted(missing)}")

    out = frame.with_columns(
        pl.col("SubjectID").cast(pl.Int64, strict=False).alias("__subject_int"),
        pl.col("Trial").cast(pl.Int64, strict=False).alias("__trial_int"),
    )
    keep = ~pl.col("__subject_int").is_in(list(OFFICIAL_EXCLUDE_SUBJECTS))
    for sid, trials in OFFICIAL_EXCLUDE_TRIALS.items():
        keep = keep & ~((pl.col("__subject_int") == sid) & pl.col("__trial_int").is_in(list(trials)))
    return out.filter(keep).drop(["__subject_int", "__trial_int"])

def validate_emotion_ids(values: Iterable[int | float | None], *, allow_null: bool = False) -> None:
    """Raise if any emotion ids fall outside the expected five-class domain."""
    valid = set(EMOTION_IDS)
    bad: list[object] = []
    for value in values:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            if not allow_null:
                bad.append(value)
            continue
        try:
            as_int = int(value)
        except (TypeError, ValueError):
            bad.append(value)
            continue
        if as_int not in valid:
            bad.append(value)
    if bad:
        preview = bad[:10]
        raise ValueError(f"Emotion ids outside {sorted(valid)}: {preview}")

### 1-a. Label mapping smoke check

In [ ]:
label_check = pl.DataFrame({"NoVideo": [1, 10, 11, 20, 21, 30, 31, 40, 41, 50], "EmotType": [1,2,3,4,5,1,2,3,4,5]})
label_check = add_target_columns(label_check)
display(label_check.to_pandas())
assert label_check["IntendedType"].to_list() == [1,1,2,2,3,3,4,4,5,5]

## 2. Metric functions

Source: `src/llamac_research/metrics.py`. Top-k, macro/weighted F1, balanced accuracy,
Cohen's kappa, confusion matrix를 계산하는 실제 구현입니다.

In [ ]:
@dataclass(frozen=True)
class ClassificationMetrics:
    """Serializable metric bundle."""

    top1_accuracy: float
    top2_accuracy: float
    top3_accuracy: float
    macro_f1: float
    weighted_f1: float
    balanced_accuracy: float
    cohen_kappa: float
    confusion_matrix: list[list[int]]
    per_class: dict[str, dict[str, float | int]]

    def to_dict(self) -> dict[str, Any]:
        return {
            "top1_accuracy": self.top1_accuracy,
            "top2_accuracy": self.top2_accuracy,
            "top3_accuracy": self.top3_accuracy,
            "macro_f1": self.macro_f1,
            "weighted_f1": self.weighted_f1,
            "balanced_accuracy": self.balanced_accuracy,
            "cohen_kappa": self.cohen_kappa,
            "confusion_matrix": self.confusion_matrix,
            "per_class": self.per_class,
        }

def _top_k_accuracy(y_true: np.ndarray, y_score: np.ndarray, labels: Sequence[int], k: int) -> float:
    if y_true.size == 0:
        return float("nan")
    label_arr = np.asarray(labels)
    k = min(k, y_score.shape[1])
    top_idx = np.argpartition(y_score, kth=y_score.shape[1] - k, axis=1)[:, -k:]
    top_labels = label_arr[top_idx]
    return float(np.mean([yt in row for yt, row in zip(y_true, top_labels, strict=False)]))

def compute_classification_metrics(
    y_true: Sequence[int],
    y_pred: Sequence[int],
    y_score: Sequence[Sequence[float]] | np.ndarray | None = None,
    *,
    labels: Sequence[int] = EMOTION_IDS,
    label_names: Sequence[str] = EMOTION_LABELS,
) -> ClassificationMetrics:
    """Compute the required LLaMAC classification metrics."""
    from sklearn.metrics import (
        balanced_accuracy_score,
        cohen_kappa_score,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
    )

    y_true_arr = np.asarray(y_true, dtype=int)
    y_pred_arr = np.asarray(y_pred, dtype=int)
    label_list = list(labels)
    if y_score is None:
        score = np.zeros((y_pred_arr.size, len(label_list)), dtype=float)
        index = {label: idx for idx, label in enumerate(label_list)}
        for row, pred in enumerate(y_pred_arr):
            if pred in index:
                score[row, index[pred]] = 1.0
    else:
        score = np.asarray(y_score, dtype=float)
    top1 = float(np.mean(y_true_arr == y_pred_arr)) if y_true_arr.size else float("nan")
    top2 = _top_k_accuracy(y_true_arr, score, label_list, 2)
    top3 = _top_k_accuracy(y_true_arr, score, label_list, 3)
    macro = float(f1_score(y_true_arr, y_pred_arr, labels=label_list, average="macro", zero_division=0))
    weighted = float(f1_score(y_true_arr, y_pred_arr, labels=label_list, average="weighted", zero_division=0))
    balanced = float(balanced_accuracy_score(y_true_arr, y_pred_arr))
    kappa = float(cohen_kappa_score(y_true_arr, y_pred_arr, labels=label_list))
    cm = confusion_matrix(y_true_arr, y_pred_arr, labels=label_list)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_arr,
        y_pred_arr,
        labels=label_list,
        zero_division=0,
    )
    per_class = {
        name: {
            "label_id": int(label),
            "precision": float(p),
            "recall": float(r),
            "f1": float(f),
            "support": int(s),
        }
        for label, name, p, r, f, s in zip(label_list, label_names, precision, recall, f1, support, strict=True)
    }
    return ClassificationMetrics(
        top1_accuracy=top1,
        top2_accuracy=top2,
        top3_accuracy=top3,
        macro_f1=macro,
        weighted_f1=weighted,
        balanced_accuracy=balanced,
        cohen_kappa=kappa,
        confusion_matrix=cm.astype(int).tolist(),
        per_class=per_class,
    )

def align_proba_columns(model_classes: Sequence[int], proba: np.ndarray, labels: Sequence[int] = EMOTION_IDS) -> np.ndarray:
    """Align estimator predict_proba columns to the canonical label order."""
    classes = [int(c) for c in model_classes]
    label_list = list(labels)
    aligned = np.zeros((proba.shape[0], len(label_list)), dtype=float)
    for src_idx, label in enumerate(classes):
        if label in label_list:
            aligned[:, label_list.index(label)] = proba[:, src_idx]
    row_sum = aligned.sum(axis=1, keepdims=True)
    missing = row_sum.squeeze() == 0
    if np.any(missing):
        aligned[missing, :] = 1.0 / len(label_list)
    else:
        aligned = aligned / row_sum
    return aligned

def metrics_summary_line(metrics: dict[str, Any]) -> str:
    """Compact human-readable metric line for CLI logs."""
    return (
        f"top1={metrics['top1_accuracy']:.4f} top2={metrics['top2_accuracy']:.4f} "
        f"top3={metrics['top3_accuracy']:.4f} macro_f1={metrics['macro_f1']:.4f} "
        f"balanced_acc={metrics['balanced_accuracy']:.4f} kappa={metrics['cohen_kappa']:.4f}"
    )

### 2-a. Metric smoke check

In [ ]:
metric_smoke = compute_classification_metrics(
    [1, 2, 3, 4, 5],
    [1, 2, 2, 4, 5],
    np.eye(5)[np.array([1, 2, 2, 4, 5]) - 1],
).to_dict()
display(pd.DataFrame([metric_smoke]).drop(columns=["confusion_matrix", "per_class"]))
assert metric_smoke["top1_accuracy"] == 0.8

## 3. Device/backend selection

Source: `src/llamac_research/device.py`. LightGBM GPU/CUDA는 build 지원 여부가 달라서
실행 시 probe 후 CPU fallback을 기록합니다.

In [ ]:
DeviceRequest = Literal["auto", "cuda", "cpu"]

@dataclass(frozen=True)
class BackendSelection:
    """Resolved backend metadata for reproducible experiment logs."""

    requested: str
    selected: str
    backend: str
    gpu_name: str | None = None
    fallback_reason: str | None = None

    def to_dict(self) -> dict[str, str | None]:
        return asdict(self)

def select_torch_device(requested: DeviceRequest = "auto") -> BackendSelection:
    """Resolve a PyTorch-style device without requiring torch at import time."""
    if requested == "cpu":
        return BackendSelection(requested=requested, selected="cpu", backend="torch")
    try:
        import torch
    except Exception as exc:  # pragma: no cover - depends on optional install
        if requested == "cuda":
            return BackendSelection(
                requested=requested,
                selected="cpu",
                backend="torch",
                fallback_reason=f"torch import failed: {exc}",
            )
        return BackendSelection(
            requested=requested,
            selected="cpu",
            backend="torch",
            fallback_reason="torch is not installed",
        )

    if torch.cuda.is_available():
        idx = torch.cuda.current_device()
        return BackendSelection(
            requested=requested,
            selected="cuda",
            backend="torch",
            gpu_name=torch.cuda.get_device_name(idx),
        )
    return BackendSelection(
        requested=requested,
        selected="cpu",
        backend="torch",
        fallback_reason="torch.cuda.is_available() is false",
    )

def _probe_lightgbm_device(device_type: str) -> tuple[bool, str | None]:
    """Return whether this LightGBM build can train with a requested device_type."""
    try:
        import lightgbm as lgb
        import numpy as np
    except Exception as exc:  # pragma: no cover - optional dependency
        return False, f"lightgbm import failed: {exc}"

    x = np.array(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
            [0.2, 0.1],
            [0.8, 0.9],
        ],
        dtype=float,
    )
    y = np.array([0, 0, 1, 1, 0, 1], dtype=int)
    params = {
        "objective": "multiclass",
        "num_class": 2,
        "metric": "multi_logloss",
        "verbosity": -1,
        "device_type": device_type,
        "num_threads": 1,
        "force_col_wise": True,
    }
    try:
        lgb.train(params, lgb.Dataset(x, label=y), num_boost_round=1)
    except Exception as exc:  # pragma: no cover - depends on local build
        return False, str(exc).splitlines()[0]
    return True, None

def select_lightgbm_device(requested: DeviceRequest = "auto") -> BackendSelection:
    """Resolve LightGBM device_type with CUDA/GPU probing and CPU fallback."""
    if requested == "cpu":
        return BackendSelection(requested=requested, selected="cpu", backend="lightgbm")

    failures: list[str] = []
    for candidate in ("cuda", "gpu"):
        ok, reason = _probe_lightgbm_device(candidate)
        if ok:
            return BackendSelection(requested=requested, selected=candidate, backend="lightgbm")
        failures.append(f"{candidate}: {reason}")

    return BackendSelection(
        requested=requested,
        selected="cpu",
        backend="lightgbm",
        fallback_reason="; ".join(failures),
    )

def select_xgboost_device(requested: DeviceRequest = "auto") -> BackendSelection:
    """Resolve XGBoost device string. XGBoost itself will still validate at fit time."""
    if requested == "cpu":
        return BackendSelection(requested=requested, selected="cpu", backend="xgboost")
    torch_selection = select_torch_device(requested="auto")
    if torch_selection.selected == "cuda":
        return BackendSelection(
            requested=requested,
            selected="cuda",
            backend="xgboost",
            gpu_name=torch_selection.gpu_name,
        )
    return BackendSelection(
        requested=requested,
        selected="cpu",
        backend="xgboost",
        fallback_reason=torch_selection.fallback_reason or "CUDA was not detected",
    )

### 3-a. Backend probe

In [ ]:
print(f"Resolving LightGBM backend for DEVICE={DEVICE!r} ...")
backend_probe = select_lightgbm_device(DEVICE)

print(f"Requested DEVICE setting: {backend_probe.requested}")
print(f"Selected LightGBM device_type: {backend_probe.selected}")
print(f"Backend: {backend_probe.backend}")
if backend_probe.gpu_name:
    print(f"GPU: {backend_probe.gpu_name}")
if backend_probe.fallback_reason:
    print(f"Fallback reason: {backend_probe.fallback_reason}")

display(pd.DataFrame([backend_probe.to_dict()]))


## 4. Feature extraction: constants, IO, robust statistics, time/signal helpers

Source: `src/llamac_research/features.py`. 이 블록은 modality summarizer들이 의존하는
공통 helper입니다.

In [ ]:
FeatureMode = Literal["all", "ppg", "ppg_rich"]
PPG_MIN_DIST_SEC = 0.35
RESP_MIN_DIST_SEC = 1.0
GSR_MIN_DIST_SEC = 1.0
try:
    from scipy.signal import butter, filtfilt, find_peaks, welch
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

@dataclass(frozen=True)
class FeatureBuildSummary:
    """Summary emitted after building a feature table."""

    rows: int
    columns: int
    participants: int
    trials: int
    feature_mode: str
    output_path: str | None

    def to_dict(self) -> dict[str, int | str | None]:
        return {
            "rows": self.rows,
            "columns": self.columns,
            "participants": self.participants,
            "trials": self.trials,
            "feature_mode": self.feature_mode,
            "output_path": self.output_path,
        }

def sanitize_name(name: str) -> str:
    """Normalize CSV column names to stable identifier-like names."""
    name = name.strip()
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

def extract_trial_number(path: str | Path) -> int | None:
    """Return the numeric trial suffix from band_12.csv style names."""
    m = re.search(r"_(\d+)\.csv$", Path(path).name)
    return int(m.group(1)) if m else None

def natural_key(path: str | Path) -> list[int | str]:
    """Natural-sort key for participant/trial filenames."""
    text = str(path)
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", text)]

def _read_csv(path: str | Path) -> pl.DataFrame:
    """Read CSV through Polars with encoding fallbacks."""
    p = Path(path)
    errors: list[str] = []
    for enc in ("utf8", "utf8-lossy", "cp949", "euc-kr"):
        try:
            df = pl.read_csv(p, encoding=enc, infer_schema_length=256, ignore_errors=True)
            return df.rename({c: sanitize_name(c) for c in df.columns})
        except Exception as exc:
            errors.append(f"{enc}: {exc}")
    raise ValueError(f"failed to read CSV {p}: {'; '.join(errors[:2])}")

def _as_float_array(values: Any) -> np.ndarray:
    """Convert a Polars series/list into a float array, coercing invalid values."""
    if isinstance(values, pl.Series):
        series = values.cast(pl.Float64, strict=False)
        return series.to_numpy().astype(float, copy=False)
    arr = np.asarray(values)
    if arr.dtype.kind in {"f", "i", "u", "b"}:
        return arr.astype(float, copy=False)
    out = np.empty(arr.shape[0], dtype=float)
    for i, item in enumerate(arr):
        try:
            out[i] = float(item)
        except (TypeError, ValueError):
            out[i] = np.nan
    return out

def robust_stats(values: Any, prefix: str) -> dict[str, float | int]:
    """Basic robust statistics used by the official baseline notebook."""
    x = _as_float_array(values)
    x = x[np.isfinite(x)]
    keys = (
        "count",
        "min",
        "max",
        "mean",
        "var",
        "std",
        "median",
        "iqr",
        "q10",
        "q90",
        "skew",
        "kurt",
        "cv",
        "rms",
        "energy",
    )
    if x.size == 0:
        return {f"{prefix}_{k}": math.nan for k in keys}

    q10, q25, q50, q75, q90 = np.quantile(x, [0.10, 0.25, 0.50, 0.75, 0.90])
    mean = float(np.mean(x))
    var = float(np.var(x, ddof=1)) if x.size > 1 else 0.0
    std = float(np.sqrt(var))
    centered = x - mean
    if std > 0 and x.size > 2:
        skew = float(np.mean((centered / std) ** 3))
        kurt = float(np.mean((centered / std) ** 4) - 3.0)
    else:
        skew = math.nan
        kurt = math.nan
    cv = float(std / mean) if mean != 0.0 else math.nan
    rms = float(np.sqrt(np.mean(x**2)))
    return {
        f"{prefix}_count": int(x.size),
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_mean": mean,
        f"{prefix}_var": var,
        f"{prefix}_std": std,
        f"{prefix}_median": float(q50),
        f"{prefix}_iqr": float(q75 - q25),
        f"{prefix}_q10": float(q10),
        f"{prefix}_q90": float(q90),
        f"{prefix}_skew": skew,
        f"{prefix}_kurt": kurt,
        f"{prefix}_cv": cv,
        f"{prefix}_rms": rms,
        f"{prefix}_energy": float(np.sum(x**2) / x.size),
    }

def parse_time_seconds(values: Any) -> np.ndarray:
    """Convert numeric or timestamp-like columns into seconds."""
    if isinstance(values, pl.Series):
        raw = values.to_list()
    else:
        raw = list(values)
    numeric = _as_float_array(raw)
    if np.isfinite(numeric).sum() >= 2:
        return numeric

    out = np.empty(len(raw), dtype=float)
    out[:] = np.nan
    for idx, item in enumerate(raw):
        if item is None:
            continue
        text = str(item).strip()
        if not text:
            continue
        # Python handles both 'YYYY-mm-dd HH:MM:SS.sss' and ISO forms here.
        try:
            out[idx] = datetime.fromisoformat(text.replace("Z", "+00:00")).timestamp()
        except ValueError:
            try:
                out[idx] = np.datetime64(text).astype("datetime64[ns]").astype("int64") / 1e9
            except Exception:
                out[idx] = np.nan
    if np.isfinite(out).sum() >= 2:
        return out
    return np.arange(len(raw), dtype=float)

def duration_and_fs(time_values: Any, n_values: int, prefix: str) -> dict[str, float]:
    """Estimate duration and sampling rate from a time column."""
    t = parse_time_seconds(time_values)
    t = t[np.isfinite(t)]
    if t.size >= 2:
        duration = float(np.max(t) - np.min(t))
        fs = float((n_values - 1) / duration) if duration > 0 and n_values > 1 else math.nan
    else:
        duration = math.nan
        fs = math.nan
    return {f"{prefix}_duration_s": duration, f"{prefix}_fs_hz": fs}

def interpolate_nans(values: np.ndarray) -> np.ndarray:
    """Linearly interpolate NaNs, edge-filling as needed."""
    y = np.asarray(values, dtype=float).copy()
    if y.size == 0:
        return y
    mask = np.isfinite(y)
    if mask.all():
        return y
    if not mask.any():
        y[:] = 0.0
        return y
    idx = np.arange(y.size)
    first = np.flatnonzero(mask)[0]
    last = np.flatnonzero(mask)[-1]
    y[:first] = y[first]
    y[last + 1 :] = y[last]
    y[~mask] = np.interp(idx[~mask], idx[mask], y[mask])
    return y

def safe_trapezoid(y: np.ndarray | None, x: np.ndarray | None) -> float:
    """Numerical integration with finite-value guards."""
    if y is None or x is None:
        return math.nan
    y_arr = np.asarray(y, dtype=float)
    x_arr = np.asarray(x, dtype=float)
    mask = np.isfinite(y_arr) & np.isfinite(x_arr)
    if mask.sum() < 2:
        return math.nan
    return float(np.trapezoid(y_arr[mask], x_arr[mask]))

def linear_slope(time_values: Any, signal_values: Any) -> float:
    """Least-squares linear slope for finite samples."""
    x = parse_time_seconds(time_values)
    y = _as_float_array(signal_values)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return math.nan
    x = x[mask]
    y = y[mask]
    x = x - np.mean(x)
    denom = float(np.dot(x, x))
    if denom == 0.0:
        return math.nan
    return float(np.dot(x, y - np.mean(y)) / denom)

def zcr(signal_values: Any) -> float:
    """Zero-crossing rate after mean-centering."""
    y = _as_float_array(signal_values)
    y = y[np.isfinite(y)]
    if y.size < 2:
        return math.nan
    centered = y - np.mean(y)
    return float(np.mean(np.diff(np.signbit(centered)) != 0))

### 4-a. Inspect extracted participant folders before feature building

In [ ]:
def _candidate_subject_dirs_for_preview(root: Path) -> list[Path]:
    seen = set()
    out = []
    for answer_path in list(root.glob("*/answer.csv")) + list(root.glob("*/*/answer.csv")):
        resolved = answer_path.parent.resolve()
        if resolved not in seen:
            seen.add(resolved)
            out.append(answer_path.parent)
    return sorted(out, key=natural_key)

subject_preview = []
for subject_dir in _candidate_subject_dirs_for_preview(DATA_ROOT):
    subject_preview.append({
        "subject_id": subject_dir.name,
        "band": len(list(subject_dir.glob("band_*.csv"))),
        "eeg": len(list(subject_dir.glob("eeg_*.csv"))),
        "resp": len(list(subject_dir.glob("respiration_*.csv"))),
    })
subject_preview = pd.DataFrame(subject_preview)
display(subject_preview.head())
display(subject_preview.describe())
assert len(subject_preview) == 108
assert subject_preview["band"].sum() == 5400

## 5. Feature extraction: filtering, peaks, spectral helpers

Source: `src/llamac_research/features.py`. PPG/RESP/EEG feature extraction이 사용하는
signal-processing helper입니다.

In [ ]:
def bandpass(values: np.ndarray, fs: float, low: float, high: float) -> np.ndarray:
    """Band-pass filter with graceful fallback when SciPy/fs are unavailable."""
    y = interpolate_nans(values)
    if not SCIPY_OK or not np.isfinite(fs) or fs <= 0 or y.size < 8:
        return y
    nyq = fs / 2.0
    high = min(high, nyq * 0.95)
    if low <= 0 or high <= low:
        return y
    try:
        b, a = butter(2, [low / nyq, high / nyq], btype="bandpass")
        return filtfilt(b, a, y)
    except Exception:
        return y

def detect_peaks_adaptive(
    values: np.ndarray,
    fs: float | None,
    min_dist_sec: float,
    prom_frac: float = 0.08,
) -> np.ndarray:
    """Adaptive peak detection used for PPG/GSR/RESP derived features."""
    y = interpolate_nans(values)
    if y.size < 3:
        return np.array([], dtype=int)
    if SCIPY_OK:
        distance = 1
        if fs is not None and np.isfinite(fs) and fs > 0:
            distance = max(1, int(round(fs * min_dist_sec)))
        spread = float(np.nanpercentile(y, 95) - np.nanpercentile(y, 5))
        prominence = spread * prom_frac if spread > 0 else None
        peaks, _ = find_peaks(y, distance=distance, prominence=prominence)
        return peaks.astype(int)

    # Small fallback: strict local maxima with minimum index spacing.
    peaks: list[int] = []
    min_dist = max(1, int(round((fs or 1.0) * min_dist_sec)))
    for i in range(1, y.size - 1):
        if y[i] > y[i - 1] and y[i] >= y[i + 1]:
            if not peaks or i - peaks[-1] >= min_dist:
                peaks.append(i)
    return np.asarray(peaks, dtype=int)

def _welch_psd(values: np.ndarray, fs: float | None) -> tuple[np.ndarray | None, np.ndarray | None]:
    y = interpolate_nans(values)
    if not SCIPY_OK or y.size < 8:
        return None, None
    fs_val = float(fs) if fs is not None and np.isfinite(fs) and fs > 0 else 256.0
    nperseg = min(256, y.size)
    try:
        freq, pxx = welch(y, fs=fs_val, nperseg=nperseg)
    except Exception:
        return None, None
    return freq, pxx

def band_power(freq: np.ndarray | None, pxx: np.ndarray | None, low: float, high: float) -> float:
    if freq is None or pxx is None:
        return math.nan
    mask = (freq >= low) & (freq < high)
    if not np.any(mask):
        return math.nan
    return safe_trapezoid(pxx[mask], freq[mask])

def spectral_entropy(freq: np.ndarray | None, pxx: np.ndarray | None) -> float:
    if freq is None or pxx is None:
        return math.nan
    power = np.asarray(pxx, dtype=float)
    power = power[np.isfinite(power) & (power > 0)]
    total = float(np.sum(power))
    if total <= 0 or power.size == 0:
        return math.nan
    p = power / total
    return float(-np.sum(p * np.log2(p)) / np.log2(p.size)) if p.size > 1 else 0.0

def hjorth_params(values: Any) -> tuple[float, float, float]:
    x = _as_float_array(values)
    x = x[np.isfinite(x)]
    if x.size < 3:
        return math.nan, math.nan, math.nan
    dx = np.diff(x)
    ddx = np.diff(dx)
    var0 = float(np.var(x))
    var1 = float(np.var(dx))
    var2 = float(np.var(ddx))
    activity = var0
    mobility = math.sqrt(var1 / var0) if var0 > 0 else math.nan
    complexity = math.sqrt(var2 / var1) / mobility if var1 > 0 and np.isfinite(mobility) and mobility > 0 else math.nan
    return activity, float(mobility), float(complexity)

def line_length(values: Any) -> float:
    x = _as_float_array(values)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return math.nan
    return float(np.sum(np.abs(np.diff(x))))

## 6. Feature extraction: PPG, band, respiration, EEG summarizers

Source: `src/llamac_research/features.py`. 여기서 `mode="ppg"`는 PPG만,
`mode="all"`은 GSR/PPG/SKT/EEG/RESP 전체를 집계합니다.

In [ ]:
def _ppg_features(df: pl.DataFrame, info: dict[str, Any], *, rich: bool = False) -> None:
    if "PPG" not in df.columns:
        return
    ppg = df["PPG"]
    if "PPG_Time" in df.columns:
        info.update(duration_and_fs(df["PPG_Time"], df.height, "Band_PPG"))
        fs = info.get("Band_PPG_fs_hz")
        t_sec = parse_time_seconds(df["PPG_Time"])
    else:
        fs = math.nan
        t_sec = np.arange(df.height, dtype=float)
    y0 = _as_float_array(ppg)
    finite_ratio = float(np.isfinite(y0).mean()) if y0.size else 0.0
    info.update(robust_stats(ppg, "Band_PPG"))

    if finite_ratio >= 0.5:
        fs_val = float(fs) if fs is not None and np.isfinite(fs) and fs > 0 else 30.0
        y = y0 - np.nanmedian(y0)
        y_f = bandpass(y, fs_val, low=0.5, high=8.0)
        peaks = detect_peaks_adaptive(y_f, fs_val, PPG_MIN_DIST_SEC, prom_frac=0.08)
        if peaks.size >= 2 and t_sec.size == y_f.size:
            ibis = np.diff(t_sec[peaks])
            ibis = ibis[np.isfinite(ibis) & (ibis > 0)]
        else:
            ibis = np.array([], dtype=float)
    else:
        ibis = np.array([], dtype=float)

    if ibis.size:
        info["Band_PPG_hr_bpm"] = float(60.0 / np.mean(ibis))
        info["Band_PPG_ibi_mean_s"] = float(np.mean(ibis))
        info["Band_PPG_ibi_sd_s"] = float(np.std(ibis, ddof=1)) if ibis.size > 1 else 0.0
        diff_ibis = np.diff(ibis)
        info["Band_PPG_rmssd_s"] = float(np.sqrt(np.mean(diff_ibis**2))) if diff_ibis.size else math.nan
        info["Band_PPG_pnn50"] = float(np.mean(np.abs(diff_ibis) > 0.05)) if diff_ibis.size else math.nan
    else:
        info["Band_PPG_hr_bpm"] = math.nan
        info["Band_PPG_ibi_mean_s"] = math.nan
        info["Band_PPG_ibi_sd_s"] = math.nan
        info["Band_PPG_rmssd_s"] = math.nan
        info["Band_PPG_pnn50"] = math.nan

    info["Band_PPG_slope"] = linear_slope(t_sec, ppg)
    info["Band_PPG_zcr"] = zcr(ppg)

    if not rich:
        return

    # PPG-only extension: morphology, derivatives, frequency, and coarse temporal dynamics.
    # These are excluded from `mode=all` so the all-channel feature table remains close to
    # the official Figshare notebook baseline.
    y_finite = y0[np.isfinite(y0)]
    if y_finite.size >= 2:
        info.update(robust_stats(np.diff(y_finite), "Band_PPG_diff"))
    else:
        info.update(robust_stats([], "Band_PPG_diff"))
    if y_finite.size >= 3:
        info.update(robust_stats(np.diff(y_finite, n=2), "Band_PPG_diff2"))
    else:
        info.update(robust_stats([], "Band_PPG_diff2"))

    fs_val = float(fs) if fs is not None and np.isfinite(fs) and fs > 0 else 30.0
    y_centered = interpolate_nans(y0 - np.nanmedian(y0)) if y0.size else y0
    peaks = detect_peaks_adaptive(bandpass(y_centered, fs_val, low=0.5, high=8.0), fs_val, PPG_MIN_DIST_SEC, prom_frac=0.08)
    info["Band_PPG_peak_count"] = int(peaks.size)
    duration_s = info.get("Band_PPG_duration_s", math.nan)
    info["Band_PPG_peak_rate_per_min"] = float(peaks.size * 60.0 / duration_s) if np.isfinite(duration_s) and duration_s > 0 else math.nan
    if peaks.size and y0.size:
        peak_values = y0[peaks]
        info.update(robust_stats(peak_values, "Band_PPG_peak_value"))
    else:
        info.update(robust_stats([], "Band_PPG_peak_value"))

    freq, pxx = _welch_psd(y_centered, fs_val)
    total_power = safe_trapezoid(pxx, freq) if freq is not None and pxx is not None else math.nan
    info["Band_PPG_total_power"] = float(total_power) if np.isfinite(total_power) else math.nan
    for band_name, low, high in (
        ("vlf", 0.04, 0.15),
        ("lf", 0.15, 0.40),
        ("resp", 0.40, 0.80),
        ("cardiac_low", 0.80, 2.00),
        ("cardiac_high", 2.00, 4.00),
        ("motion", 4.00, 8.00),
    ):
        bp = band_power(freq, pxx, low, high)
        info[f"Band_PPG_{band_name}_power_abs"] = bp
        info[f"Band_PPG_{band_name}_power_rel"] = bp / total_power if np.isfinite(bp) and np.isfinite(total_power) and total_power > 0 else math.nan
    if freq is not None and pxx is not None and pxx.size:
        valid = np.isfinite(freq) & np.isfinite(pxx)
        info["Band_PPG_dominant_hz"] = float(freq[valid][np.argmax(pxx[valid])]) if np.any(valid) else math.nan
        info["Band_PPG_spec_entropy"] = spectral_entropy(freq, pxx)
    else:
        info["Band_PPG_dominant_hz"] = math.nan
        info["Band_PPG_spec_entropy"] = math.nan

    if y0.size >= 8:
        edges = np.linspace(0, y0.size, 5, dtype=int)
        for idx in range(4):
            lo, hi = edges[idx], edges[idx + 1]
            seg = y0[lo:hi]
            seg_t = t_sec[lo:hi] if t_sec.size == y0.size else np.arange(seg.size, dtype=float)
            finite_seg = seg[np.isfinite(seg)]
            info[f"Band_PPG_seg{idx + 1}_mean"] = float(np.mean(finite_seg)) if finite_seg.size else math.nan
            info[f"Band_PPG_seg{idx + 1}_std"] = float(np.std(finite_seg, ddof=1)) if finite_seg.size > 1 else (0.0 if finite_seg.size else math.nan)
            info[f"Band_PPG_seg{idx + 1}_slope"] = linear_slope(seg_t, seg)
    else:
        for idx in range(4):
            info[f"Band_PPG_seg{idx + 1}_mean"] = math.nan
            info[f"Band_PPG_seg{idx + 1}_std"] = math.nan
            info[f"Band_PPG_seg{idx + 1}_slope"] = math.nan

def summarize_band_csv(path: str | Path, mode: FeatureMode = "all") -> dict[str, Any]:
    """Summarize one band_*.csv file."""
    df = _read_csv(path)
    info: dict[str, Any] = {}

    if mode == "all" and "GSR" in df.columns:
        info.update(robust_stats(df["GSR"], "Band_GSR"))
        gsr_tcol = "GSR_Time" if "GSR_Time" in df.columns else None
        if gsr_tcol:
            info.update(duration_and_fs(df[gsr_tcol], df.height, "Band_GSR"))
            fs_gsr = info.get("Band_GSR_fs_hz")
        else:
            fs_gsr = math.nan
        gsr = _as_float_array(df["GSR"])
        if np.isfinite(gsr).sum() >= 3:
            gsr_i = interpolate_nans(gsr)
            peaks = detect_peaks_adaptive(gsr_i, fs_gsr, GSR_MIN_DIST_SEC)
            if peaks.size:
                base = np.nanpercentile(gsr_i, 5)
                top = np.nanpercentile(gsr_i, 95)
                amp = gsr_i[peaks] - base
                threshold = (top - base) * 0.1
                info["Band_GSR_scr_count"] = int(np.sum(amp > threshold))
                info["Band_GSR_peak_amp_mean"] = float(np.nanmean(amp)) if amp.size else math.nan
                info["Band_GSR_peak_amp_max"] = float(np.nanmax(amp)) if amp.size else math.nan
            else:
                info["Band_GSR_scr_count"] = 0
                info["Band_GSR_peak_amp_mean"] = math.nan
                info["Band_GSR_peak_amp_max"] = math.nan
        else:
            info["Band_GSR_scr_count"] = 0
            info["Band_GSR_peak_amp_mean"] = math.nan
            info["Band_GSR_peak_amp_max"] = math.nan
        info["Band_GSR_slope"] = linear_slope(df[gsr_tcol] if gsr_tcol else np.arange(df.height), df["GSR"])
        info["Band_GSR_zcr"] = zcr(df["GSR"])

    _ppg_features(df, info, rich=(mode == "ppg_rich"))

    if mode == "all" and "SKT" in df.columns:
        info.update(robust_stats(df["SKT"], "Band_SKT"))
        skt_tcol = "SKT_Time" if "SKT_Time" in df.columns else None
        if skt_tcol:
            info.update(duration_and_fs(df[skt_tcol], df.height, "Band_SKT"))
        info["Band_SKT_slope"] = linear_slope(df[skt_tcol] if skt_tcol else np.arange(df.height), df["SKT"])
    return info

def summarize_resp_csv(path: str | Path) -> dict[str, Any]:
    """Summarize one respiration_*.csv file."""
    df = _read_csv(path)
    info: dict[str, Any] = {}
    time_col = next((c for c in ("Time", "Timestamp") if c in df.columns), None)
    force_col = next((c for c in df.columns if c.lower().startswith("force")), None)
    if force_col is None:
        return info

    info.update(robust_stats(df[force_col], "Resp_Force"))
    if time_col:
        info.update(duration_and_fs(df[time_col], df.height, "Resp"))
        t_sec = parse_time_seconds(df[time_col])
        fs = info.get("Resp_fs_hz")
    else:
        t_sec = np.arange(df.height, dtype=float)
        fs = math.nan

    y0 = _as_float_array(df[force_col])
    if np.isfinite(y0).sum() >= 3:
        peaks = detect_peaks_adaptive(interpolate_nans(y0), fs, RESP_MIN_DIST_SEC, prom_frac=0.08)
        if peaks.size >= 2 and t_sec.size == y0.size:
            ibis = np.diff(t_sec[peaks])
            ibis = ibis[np.isfinite(ibis) & (ibis > 0)]
        else:
            ibis = np.array([], dtype=float)
    else:
        ibis = np.array([], dtype=float)

    if ibis.size:
        info["Resp_breath_bpm"] = float(60.0 / np.mean(ibis))
        info["Resp_ibi_mean_s"] = float(np.mean(ibis))
        info["Resp_ibi_sd_s"] = float(np.std(ibis, ddof=1)) if ibis.size > 1 else 0.0
    else:
        info["Resp_breath_bpm"] = math.nan
        info["Resp_ibi_mean_s"] = math.nan
        info["Resp_ibi_sd_s"] = math.nan
    info["Resp_slope"] = linear_slope(t_sec, df[force_col])
    info["Resp_zcr"] = zcr(df[force_col])
    return info

def summarize_eeg_csv(path: str | Path) -> dict[str, Any]:
    """Summarize one eeg_*.csv file."""
    df = _read_csv(path)
    info: dict[str, Any] = {}
    tcol = next((c for c in ("Timestamp", "Time", "TS") if c in df.columns), None)
    if tcol:
        info.update(duration_and_fs(df[tcol], df.height, "EEG"))
        fs = info.get("EEG_fs_hz")
        t_sec = parse_time_seconds(df[tcol])
    else:
        fs = math.nan
        t_sec = np.arange(df.height, dtype=float)

    bands = {
        "delta": (1, 4),
        "theta": (4, 7),
        "alpha": (8, 13),
        "beta": (13, 30),
        "gamma": (30, 45),
    }
    for col in df.columns:
        if col == tcol:
            continue
        values = _as_float_array(df[col])
        prefix = f"EEG_{col}"
        info.update(robust_stats(df[col], prefix))
        activity, mobility, complexity = hjorth_params(values)
        info[f"{prefix}_hjorth_activity"] = activity
        info[f"{prefix}_hjorth_mobility"] = mobility
        info[f"{prefix}_hjorth_complexity"] = complexity
        info[f"{prefix}_line_length"] = line_length(values)
        info[f"{prefix}_zcr"] = zcr(values)
        info[f"{prefix}_slope"] = linear_slope(t_sec, values)

        freq, pxx = _welch_psd(values, fs)
        total_power = safe_trapezoid(pxx, freq) if freq is not None and pxx is not None else math.nan
        info[f"{prefix}_total_power"] = float(total_power) if np.isfinite(total_power) else math.nan
        for band_name, (low, high) in bands.items():
            bp = band_power(freq, pxx, low, high)
            info[f"{prefix}_{band_name}_abs"] = bp
            info[f"{prefix}_{band_name}_rel"] = bp / total_power if np.isfinite(bp) and np.isfinite(total_power) and total_power > 0 else math.nan
        if freq is not None and pxx is not None:
            mask = (freq >= 8) & (freq <= 13)
            info[f"{prefix}_alpha_peak_hz"] = float(freq[mask][np.argmax(pxx[mask])]) if np.any(mask) else math.nan
            info[f"{prefix}_spec_entropy"] = spectral_entropy(freq, pxx)
        else:
            info[f"{prefix}_alpha_peak_hz"] = math.nan
            info[f"{prefix}_spec_entropy"] = math.nan
    return info

### 6-a. Single-trial feature inspection

In [ ]:
sample_subject = _candidate_subject_dirs_for_preview(DATA_ROOT)[0]
sample_band = sorted(sample_subject.glob("band_*.csv"), key=natural_key)[0]
sample_ppg_features = summarize_band_csv(sample_band, mode="ppg")
sample_all_band_features = summarize_band_csv(sample_band, mode="all")
print(sample_band)
display(pd.DataFrame([
    {"mode": "ppg", "feature_count": len(sample_ppg_features), "example_keys": list(sample_ppg_features)[:8]},
    {"mode": "all-band", "feature_count": len(sample_all_band_features), "example_keys": list(sample_all_band_features)[:8]},
]))
assert all(k.startswith("Band_PPG_") for k in sample_ppg_features)

## 7. Feature extraction: participant processing and feature table builder

Source: `src/llamac_research/features.py`. 이 셀이 실제 all-channel/PPG feature table을
만드는 함수들을 정의합니다.

In [ ]:
def read_answer_csv(subject_dir: str | Path, subject_id: str | int | None = None) -> pl.DataFrame:
    """Read one participant answer.csv and add SubjectID if absent."""
    sdir = Path(subject_dir)
    answer_path = next((sdir / name for name in ("answer.csv", "Answer.csv", "ANSWER.csv") if (sdir / name).is_file()), None)
    if answer_path is None:
        raise FileNotFoundError(f"answer.csv not found under {sdir}")
    frame = _read_csv(answer_path)
    if "Trial" not in frame.columns:
        raise ValueError(f"Trial column missing in {answer_path}")
    if "SubjectID" not in frame.columns:
        frame = frame.with_columns(pl.lit(str(subject_id or sdir.name)).alias("SubjectID"))
    frame = frame.select([c for c in ANSWER_COLUMNS if c in frame.columns])
    return add_target_columns(frame)

def process_subject(subject_dir: str | Path, mode: FeatureMode = "all") -> pl.DataFrame:
    """Build merged label+feature rows for one participant directory."""
    sdir = Path(subject_dir)
    subject_id = sdir.name
    answer = read_answer_csv(sdir, subject_id=subject_id)
    trial_map: dict[int, dict[str, list[Path]]] = {}
    patterns = ["band_*.csv"] if mode in {"ppg", "ppg_rich"} else ["band_*.csv", "eeg_*.csv", "respiration_*.csv"]
    for pattern in patterns:
        for path in sorted(sdir.glob(pattern), key=natural_key):
            trial = extract_trial_number(path)
            if trial is None:
                continue
            bucket = trial_map.setdefault(trial, {"band": [], "eeg": [], "resp": []})
            lower = path.name.lower()
            if lower.startswith("band_"):
                bucket["band"].append(path)
            elif lower.startswith("eeg_"):
                bucket["eeg"].append(path)
            elif lower.startswith("respiration_"):
                bucket["resp"].append(path)

    rows: list[dict[str, Any]] = []
    trials = answer["Trial"].cast(pl.Int64, strict=False).drop_nulls().to_list()
    for trial in sorted(int(t) for t in trials):
        feat: dict[str, Any] = {"Trial": int(trial)}
        paths = trial_map.get(int(trial), {"band": [], "eeg": [], "resp": []})
        for path in paths["band"]:
            feat.update(summarize_band_csv(path, mode=mode))
        if mode == "all":
            for path in paths["eeg"]:
                feat.update(summarize_eeg_csv(path))
            for path in paths["resp"]:
                feat.update(summarize_resp_csv(path))
        rows.append(feat)
    feature_frame = pl.DataFrame(rows) if rows else pl.DataFrame({"Trial": []})
    feature_frame = feature_frame.with_columns(pl.col("Trial").cast(pl.Int64, strict=False))
    answer = answer.with_columns(pl.col("Trial").cast(pl.Int64, strict=False))
    return answer.join(feature_frame, on="Trial", how="left")

def _process_subject_worker(args: tuple[str, str]) -> pl.DataFrame:
    subject_dir, mode = args
    return process_subject(subject_dir, mode=mode)  # type: ignore[arg-type]

def discover_subject_dirs(data_root: str | Path, limit_subjects: int | None = None) -> list[Path]:
    """Discover participant folders, including archives that extracted one level deep.

    Most LLaMAC zip files extract directly to `data/extracted/<id>/answer.csv`,
    but at least one archive can appear as `data/extracted/<id>/<id>/answer.csv`.
    The feature builder treats the directory containing `answer.csv` as the
    participant directory and de-duplicates by resolved path.
    """
    root = Path(data_root)
    subject_dirs: list[Path] = []
    seen: set[Path] = set()
    for answer_path in root.glob("*/answer.csv"):
        parent = answer_path.parent.resolve()
        if parent not in seen:
            seen.add(parent)
            subject_dirs.append(answer_path.parent)
    for answer_path in root.glob("*/*/answer.csv"):
        parent = answer_path.parent.resolve()
        if parent not in seen:
            seen.add(parent)
            subject_dirs.append(answer_path.parent)
    subject_dirs = sorted(subject_dirs, key=natural_key)
    if limit_subjects is not None:
        subject_dirs = subject_dirs[:limit_subjects]
    return subject_dirs

def build_feature_table(
    data_root: str | Path = "data/extracted",
    *,
    mode: FeatureMode = "all",
    limit_subjects: int | None = None,
    workers: int = 1,
    output_path: str | Path | None = None,
) -> tuple[pl.DataFrame, FeatureBuildSummary]:
    """Build a trial-wise feature table from extracted LLaMAC participant folders."""
    subject_dirs = discover_subject_dirs(data_root, limit_subjects=limit_subjects)
    if not subject_dirs:
        raise FileNotFoundError(f"No participant folders with answer.csv under {data_root}")

    frames: list[pl.DataFrame] = []
    if workers <= 1:
        for idx, subject_dir in enumerate(subject_dirs, start=1):
            print(f"[{idx}/{len(subject_dirs)}] subject={subject_dir.name} mode={mode}", flush=True)
            frames.append(process_subject(subject_dir, mode=mode))
    else:
        args = [(str(path), mode) for path in subject_dirs]
        with ProcessPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(_process_subject_worker, arg): Path(arg[0]).name for arg in args}
            for idx, future in enumerate(as_completed(futures), start=1):
                sid = futures[future]
                try:
                    frames.append(future.result())
                    print(f"[{idx}/{len(subject_dirs)}] subject={sid} done", flush=True)
                except Exception as exc:
                    warnings.warn(f"subject {sid} failed: {exc}")

    if not frames:
        raise RuntimeError("No feature frames were produced")
    merged = pl.concat(frames, how="diagonal_relaxed")
    first_cols = [c for c in [*ANSWER_COLUMNS, "IntendedType", "ReportedType"] if c in merged.columns]
    other_cols = [c for c in merged.columns if c not in first_cols]
    merged = merged.select(first_cols + sorted(other_cols))

    if output_path is not None:
        out = Path(output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        if out.suffix.lower() == ".parquet":
            merged.write_parquet(out)
        else:
            merged.write_csv(out)
    summary = FeatureBuildSummary(
        rows=merged.height,
        columns=merged.width,
        participants=merged.select(pl.col("SubjectID").n_unique()).item() if "SubjectID" in merged.columns else 0,
        trials=merged.select(pl.col("Trial").n_unique()).item() if "Trial" in merged.columns else 0,
        feature_mode=mode,
        output_path=str(output_path) if output_path is not None else None,
    )
    return merged, summary

def read_feature_table(path: str | Path) -> pl.DataFrame:
    """Read a generated feature table from CSV or parquet."""
    p = Path(path)
    if p.suffix.lower() == ".parquet":
        return pl.read_parquet(p)
    return pl.read_csv(p, infer_schema_length=1024, ignore_errors=True)

def write_summary_csv(rows: Sequence[dict[str, Any]], output_path: str | Path) -> None:
    """Write compact dict rows as CSV without requiring pandas."""
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    fieldnames: list[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with out.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

### 7-a. Build or load all-channel and PPG-only feature tables

In [ ]:
def ensure_feature_table(mode: str, path: Path) -> pl.DataFrame:
    if path.exists() and not REBUILD_FEATURES:
        frame = read_feature_table(path)
        print(f"[load] {mode}: {path} shape={frame.shape}")
    else:
        frame, summary = build_feature_table(DATA_ROOT, mode=mode, workers=WORKERS, output_path=path)
        print(f"[built] {summary}")
    return frame

all_features = ensure_feature_table("all", FEATURE_ALL)
ppg_features = ensure_feature_table("ppg", FEATURE_PPG)

display(pd.DataFrame([
    {"mode": "all", "rows": all_features.height, "columns": all_features.width},
    {"mode": "ppg", "rows": ppg_features.height, "columns": ppg_features.width},
]))
assert all_features.height == 5400
assert ppg_features.height == 5400

## 8. Modeling harness: config, feature matrix, fold preprocessing, splitters

Source: `src/llamac_research/modeling.py`. 여기서 self-report leakage columns를 제거하고,
participant-grouped split을 준비합니다.

In [ ]:
__version__ = "0.1.0"
FeatureSet = Literal["all", "ppg"]
SplitStrategy = Literal["grouped", "stratified"]
ModelName = Literal[
    "lightgbm",
    "logistic_regression",
    "svc_rbf",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
]
MODEL_NAMES: tuple[str, ...] = (
    "lightgbm",
    "logistic_regression",
    "svc_rbf",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
)

@dataclass(frozen=True)
class ExperimentConfig:
    """Serializable model-evaluation configuration."""

    feature_path: str
    model_name: str = "lightgbm"
    feature_set: str = "all"
    target: str = "reported"
    split_strategy: str = "grouped"
    n_splits: int = 5
    seed: int = 42
    device: str = "auto"
    apply_official_exclusions: bool = False
    max_rows: int | None = None
    output_dir: str = "artifacts/results"
    model_params: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class FeatureMatrix:
    """Prepared raw feature matrix before fold-specific imputation."""

    x: np.ndarray
    y: np.ndarray
    groups: np.ndarray
    feature_names: list[str]
    label_distribution: dict[str, int]

def git_commit() -> str | None:
    """Return current git commit hash when available without shelling out."""
    try:
        head_path = Path(".git") / "HEAD"
        head = head_path.read_text(encoding="utf-8").strip()
        if head.startswith("ref: "):
            ref_path = Path(".git") / head.split(" ", 1)[1]
            return ref_path.read_text(encoding="utf-8").strip()
        return head
    except Exception:
        return None

def file_sha256(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def select_feature_columns(frame: pl.DataFrame, feature_set: FeatureSet = "all") -> list[str]:
    """Return candidate numeric feature columns with no questionnaire leakage."""
    leakage = {"SubjectID", "Trial", *SELF_REPORT_COLUMNS}
    if feature_set == "ppg":
        return [c for c in frame.columns if c.startswith("Band_PPG_")]
    if feature_set != "all":
        raise ValueError(f"Unsupported feature_set={feature_set!r}; use 'all' or 'ppg'.")
    out: list[str] = []
    for col, dtype in zip(frame.columns, frame.dtypes, strict=True):
        if col in leakage:
            continue
        if dtype.is_numeric():
            out.append(col)
    return out

def load_feature_matrix(
    feature_path: str | Path,
    *,
    feature_set: FeatureSet,
    target: str,
    apply_official_exclusions: bool = False,
    max_rows: int | None = None,
) -> FeatureMatrix:
    """Load feature table and resolve X/y/groups arrays."""
    target_spec = resolve_target(target)
    frame = read_feature_table(feature_path)
    if apply_official_exclusions:
        frame = filter_official_valid_trials(frame)
    if target_spec.column not in frame.columns:
        raise ValueError(f"{target_spec.column} is missing from {feature_path}; rebuild features with target columns.")
    if "SubjectID" not in frame.columns:
        raise ValueError("SubjectID column is required for participant-grouped splits")
    feature_names = select_feature_columns(frame, feature_set=feature_set)
    if not feature_names:
        raise ValueError(f"No feature columns found for feature_set={feature_set!r}")

    frame = frame.with_columns(pl.col(target_spec.column).cast(pl.Int64, strict=False))
    frame = frame.filter(pl.col(target_spec.column).is_in(EMOTION_IDS))
    if max_rows is not None:
        frame = frame.head(max_rows)
    feature_frame = frame.select([pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in feature_names])
    x = feature_frame.to_numpy().astype(float, copy=False)
    y = frame[target_spec.column].to_numpy().astype(int, copy=False)
    groups = frame["SubjectID"].cast(pl.Utf8).to_numpy()
    labels, counts = np.unique(y, return_counts=True)
    distribution = {str(int(label)): int(count) for label, count in zip(labels, counts, strict=True)}
    return FeatureMatrix(x=x, y=y, groups=groups, feature_names=feature_names, label_distribution=distribution)

def _fold_transform(
    x_train: np.ndarray,
    x_test: np.ndarray,
    feature_names: Sequence[str],
    *,
    max_nan_ratio: float = 0.5,
) -> tuple[np.ndarray, np.ndarray, list[str], dict[str, Any]]:
    """Drop unstable columns and median-impute using training-fold statistics only."""
    x_tr = np.asarray(x_train, dtype=float).copy()
    x_te = np.asarray(x_test, dtype=float).copy()
    x_tr[~np.isfinite(x_tr)] = np.nan
    x_te[~np.isfinite(x_te)] = np.nan
    nan_ratio = np.mean(np.isnan(x_tr), axis=0)
    keep = nan_ratio < max_nan_ratio
    # Drop constants after considering finite training values.
    for idx in range(x_tr.shape[1]):
        if not keep[idx]:
            continue
        finite = x_tr[:, idx][np.isfinite(x_tr[:, idx])]
        if finite.size == 0 or np.unique(finite).size <= 1:
            keep[idx] = False
    if not np.any(keep):
        raise ValueError("All feature columns were dropped by fold preprocessing")
    x_tr = x_tr[:, keep]
    x_te = x_te[:, keep]
    selected_names = [name for name, flag in zip(feature_names, keep, strict=True) if flag]
    medians = np.nanmedian(x_tr, axis=0)
    medians[~np.isfinite(medians)] = 0.0
    train_nan = np.isnan(x_tr)
    test_nan = np.isnan(x_te)
    x_tr[train_nan] = np.take(medians, np.where(train_nan)[1])
    x_te[test_nan] = np.take(medians, np.where(test_nan)[1])
    info = {
        "input_features": len(feature_names),
        "selected_features": len(selected_names),
        "dropped_features": int(len(feature_names) - len(selected_names)),
    }
    return x_tr, x_te, selected_names, info

def make_splitter(strategy: SplitStrategy, n_splits: int, seed: int):
    """Create sklearn splitter."""
    if strategy == "grouped":
        from sklearn.model_selection import StratifiedGroupKFold

        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    if strategy == "stratified":
        from sklearn.model_selection import StratifiedKFold

        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    raise ValueError(f"Unsupported split_strategy={strategy!r}")

def split_indices(matrix: FeatureMatrix, strategy: SplitStrategy, n_splits: int, seed: int):
    splitter = make_splitter(strategy, n_splits=n_splits, seed=seed)
    if strategy == "grouped":
        yield from splitter.split(matrix.x, matrix.y, matrix.groups)
    else:
        yield from splitter.split(matrix.x, matrix.y)

### 8-a. Matrix diagnostics

In [ ]:
all_matrix = load_feature_matrix(FEATURE_ALL, feature_set="all", target="reported")
ppg_matrix = load_feature_matrix(FEATURE_PPG, feature_set="ppg", target="reported")
display(pd.DataFrame([
    {"variant": "all", "rows": all_matrix.y.size, "groups": len(np.unique(all_matrix.groups)), "features": len(all_matrix.feature_names), "labels": all_matrix.label_distribution},
    {"variant": "ppg", "rows": ppg_matrix.y.size, "groups": len(np.unique(ppg_matrix.groups)), "features": len(ppg_matrix.feature_names), "labels": ppg_matrix.label_distribution},
]))

## 9. Modeling harness: estimator factory, prediction, CV execution

Source: `src/llamac_research/modeling.py`. LightGBM parameters and CV loop are visible here.

In [ ]:
def _backend_for_model(model_name: str, requested_device: DeviceRequest) -> BackendSelection:
    if model_name == "lightgbm":
        return select_lightgbm_device(requested_device)
    if model_name == "xgboost":
        return select_xgboost_device(requested_device)
    return BackendSelection(requested=requested_device, selected="cpu", backend=model_name)

def create_estimator(
    model_name: ModelName,
    *,
    seed: int,
    device_selection: BackendSelection,
    params: dict[str, Any] | None = None,
):
    """Instantiate a supported classifier."""
    params = dict(params or {})
    if model_name == "lightgbm":
        from lightgbm import LGBMClassifier

        base = {
            "n_estimators": 400,
            "learning_rate": 0.05,
            "max_depth": -1,
            "num_leaves": 63,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.0,
            "reg_lambda": 0.1,
            "min_child_samples": 20,
            "min_split_gain": 0.0,
            "class_weight": "balanced",
            "random_state": seed,
            "n_jobs": 1,
            "deterministic": True,
            "force_row_wise": True,
            "verbosity": -1,
            "device_type": device_selection.selected,
        }
        base.update(params)
        return LGBMClassifier(**base)

    if model_name == "logistic_regression":
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler

        base = {"max_iter": 2000, "class_weight": "balanced", "random_state": seed}
        base.update(params)
        return make_pipeline(StandardScaler(), LogisticRegression(**base))

    if model_name == "svc_rbf":
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.svm import SVC

        base = {"C": 3.0, "gamma": "scale", "class_weight": "balanced", "probability": True, "random_state": seed}
        base.update(params)
        return make_pipeline(StandardScaler(), SVC(**base))

    if model_name == "random_forest":
        from sklearn.ensemble import RandomForestClassifier

        base = {
            "n_estimators": 500,
            "max_features": "sqrt",
            "min_samples_leaf": 1,
            "class_weight": "balanced_subsample",
            "random_state": seed,
            "n_jobs": -1,
        }
        base.update(params)
        return RandomForestClassifier(**base)

    if model_name == "extra_trees":
        from sklearn.ensemble import ExtraTreesClassifier

        base = {
            "n_estimators": 700,
            "max_features": "sqrt",
            "min_samples_leaf": 1,
            "class_weight": "balanced",
            "random_state": seed,
            "n_jobs": -1,
        }
        base.update(params)
        return ExtraTreesClassifier(**base)

    if model_name == "hist_gradient_boosting":
        from sklearn.ensemble import HistGradientBoostingClassifier

        base = {"learning_rate": 0.05, "max_iter": 300, "l2_regularization": 0.01, "random_state": seed}
        base.update(params)
        return HistGradientBoostingClassifier(**base)

    if model_name == "xgboost":
        from xgboost import XGBClassifier

        base = {
            "n_estimators": 400,
            "learning_rate": 0.05,
            "max_depth": 4,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
            "objective": "multi:softprob",
            "eval_metric": "mlogloss",
            "random_state": seed,
            "n_jobs": 1,
            "device": device_selection.selected,
        }
        base.update(params)
        return XGBClassifier(**base)

    raise ValueError(f"Unsupported model_name={model_name!r}")

def _predict_scores(model: Any, x: np.ndarray, labels: Sequence[int] = EMOTION_IDS) -> tuple[np.ndarray, np.ndarray]:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="X does not have valid feature names.*")
        pred = model.predict(x)
    pred_arr = np.asarray(pred, dtype=int)
    if hasattr(model, "predict_proba"):
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="X does not have valid feature names.*")
            proba = np.asarray(model.predict_proba(x), dtype=float)
        classes = getattr(model, "classes_", labels)
        if not hasattr(model, "classes_") and hasattr(model, "named_steps"):
            # sklearn Pipeline exposes classes_ on the final estimator only in some versions.
            classes = getattr(model.steps[-1][1], "classes_", labels)
        scores = align_proba_columns(classes, proba, labels=labels)
    else:
        scores = None
    if scores is None:
        scores = align_proba_columns(labels, np.eye(len(labels))[np.searchsorted(labels, pred_arr)], labels=labels)
    return pred_arr, scores

def run_cross_validated_experiment(config: ExperimentConfig) -> dict[str, Any]:
    """Run a CV experiment and return a JSON-serializable result bundle."""
    start = time.time()
    if config.model_name not in MODEL_NAMES:
        raise ValueError(f"Unsupported model {config.model_name!r}; choices={MODEL_NAMES}")
    matrix = load_feature_matrix(
        config.feature_path,
        feature_set=config.feature_set,  # type: ignore[arg-type]
        target=config.target,
        apply_official_exclusions=config.apply_official_exclusions,
        max_rows=config.max_rows,
    )
    if len(np.unique(matrix.y)) < 2:
        raise ValueError("Need at least two target classes for classification")
    backend = _backend_for_model(config.model_name, config.device)  # type: ignore[arg-type]
    y_true_all: list[int] = []
    y_pred_all: list[int] = []
    score_all: list[np.ndarray] = []
    fold_summaries: list[dict[str, Any]] = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        split_indices(matrix, config.split_strategy, config.n_splits, config.seed), start=1
    ):
        x_train, x_test, selected_names, transform_info = _fold_transform(
            matrix.x[train_idx], matrix.x[test_idx], matrix.feature_names
        )
        y_train = matrix.y[train_idx]
        y_test = matrix.y[test_idx]
        model = create_estimator(
            config.model_name,  # type: ignore[arg-type]
            seed=config.seed + fold_idx,
            device_selection=backend,
            params=config.model_params,
        )
        fit_start = time.time()
        if config.model_name == "lightgbm":
            try:
                from lightgbm.callback import early_stopping, log_evaluation

                model.fit(
                    x_train,
                    y_train,
                    eval_set=[(x_test, y_test)],
                    eval_metric="multi_logloss",
                    callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)],
                )
            except TypeError:
                model.fit(x_train, y_train)
        else:
            model.fit(x_train, y_train)
        y_pred, scores = _predict_scores(model, x_test)
        fold_metrics = compute_classification_metrics(y_test, y_pred, scores).to_dict()
        y_true_all.extend(y_test.tolist())
        y_pred_all.extend(y_pred.tolist())
        score_all.append(scores)
        fold_summaries.append(
            {
                "fold": fold_idx,
                "train_rows": int(train_idx.size),
                "test_rows": int(test_idx.size),
                "train_groups": int(np.unique(matrix.groups[train_idx]).size),
                "test_groups": int(np.unique(matrix.groups[test_idx]).size),
                "fit_seconds": round(time.time() - fit_start, 3),
                **transform_info,
                "metrics": fold_metrics,
            }
        )
        print(f"[fold {fold_idx}] {metrics_summary_line(fold_metrics)}", flush=True)

    all_scores = np.vstack(score_all)
    overall_metrics = compute_classification_metrics(y_true_all, y_pred_all, all_scores).to_dict()
    elapsed = time.time() - start
    result = {
        "schema_version": 1,
        "created_at_unix": int(time.time()),
        "elapsed_seconds": round(elapsed, 3),
        "config": asdict(config),
        "environment": {
            "llamac_research_version": __version__,
            "python": platform.python_version(),
            "platform": platform.platform(),
            "git_commit": git_commit(),
            "feature_file_sha256": file_sha256(config.feature_path),
        },
        "backend": backend.to_dict(),
        "data": {
            "rows": int(matrix.y.size),
            "candidate_features": int(len(matrix.feature_names)),
            "label_distribution": matrix.label_distribution,
            "groups": int(np.unique(matrix.groups).size),
        },
        "folds": fold_summaries,
        "metrics": overall_metrics,
    }
    return result

def default_result_path(config: ExperimentConfig, result: dict[str, Any]) -> Path:
    """Build deterministic-ish output path from config and timestamp."""
    stamp = time.strftime("%Y%m%d-%H%M%S", time.localtime(result["created_at_unix"]))
    official = "official" if config.apply_official_exclusions else "allsubjects"
    name = f"{stamp}_{config.model_name}_{config.feature_set}_{config.target}_{config.split_strategy}_{official}.json"
    return Path(config.output_dir) / name

def save_experiment_result(result: dict[str, Any], output_path: str | Path) -> None:
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(result, indent=2, sort_keys=True), encoding="utf-8")

def run_and_save(config: ExperimentConfig, output_path: str | Path | None = None) -> Path:
    result = run_cross_validated_experiment(config)
    out = Path(output_path) if output_path is not None else default_result_path(config, result)
    save_experiment_result(result, out)
    print(f"[result] {metrics_summary_line(result['metrics'])}", flush=True)
    print(f"[saved] {out}", flush=True)
    return out

## 10. Run or load LightGBM baseline experiments

`RERUN_EXPERIMENTS=True`로 바꾸면 아래 셀이 model fit을 다시 실행합니다. 기본값은 이미 생성된
result JSON을 읽어서 빠르게 결과를 검토합니다.

In [ ]:
BASELINE_SPECS = {
    "all_reported_grouped": (
        ExperimentConfig(str(FEATURE_ALL), "lightgbm", "all", "reported", "grouped", FOLDS, SEED, DEVICE, False, None, str(RESULTS)),
        RESULTS / "lightgbm_all_reported_grouped.json",
    ),
    "ppg_reported_grouped": (
        ExperimentConfig(str(FEATURE_PPG), "lightgbm", "ppg", "reported", "grouped", FOLDS, SEED, "auto", False, None, str(RESULTS)),
        RESULTS / "lightgbm_ppg_reported_grouped.json",
    ),
    "all_reported_official_stratified": (
        ExperimentConfig(str(FEATURE_ALL), "lightgbm", "all", "reported", "stratified", FOLDS, SEED, DEVICE, True, None, str(RESULTS)),
        RESULTS / "lightgbm_all_reported_official_stratified.json",
    ),
    "all_intended_official_stratified": (
        ExperimentConfig(str(FEATURE_ALL), "lightgbm", "all", "intended", "stratified", FOLDS, SEED, DEVICE, True, None, str(RESULTS)),
        RESULTS / "lightgbm_all_intended_official_stratified.json",
    ),
    "ppg_reported_official_stratified": (
        ExperimentConfig(str(FEATURE_PPG), "lightgbm", "ppg", "reported", "stratified", FOLDS, SEED, DEVICE, True, None, str(RESULTS)),
        RESULTS / "lightgbm_ppg_reported_official_stratified.json",
    ),
}

def run_or_load_result(label: str, config: ExperimentConfig, path: Path) -> dict:
    if path.exists() and not RERUN_EXPERIMENTS:
        result = json.loads(path.read_text())
        print(f"[load] {label}: {metrics_summary_line(result['metrics'])}")
    else:
        result = run_cross_validated_experiment(config)
        save_experiment_result(result, path)
        print(f"[saved] {label}: {path}")
    return result

baseline_results = {label: run_or_load_result(label, config, path) for label, (config, path) in BASELINE_SPECS.items()}

## 11. Compare metrics and inspect degradation

In [ ]:
def result_row(label: str, result: dict) -> dict:
    cfg, data, m = result["config"], result["data"], result["metrics"]
    return {
        "label": label,
        "model": cfg["model_name"],
        "features": cfg["feature_set"],
        "target": cfg["target"],
        "split": cfg["split_strategy"],
        "official_exclusions": cfg["apply_official_exclusions"],
        "rows": data["rows"],
        "feature_count": data["candidate_features"],
        "backend": result["backend"]["selected"],
        "top1": m["top1_accuracy"],
        "top2": m["top2_accuracy"],
        "top3": m["top3_accuracy"],
        "macro_f1": m["macro_f1"],
        "weighted_f1": m["weighted_f1"],
        "balanced_acc": m["balanced_accuracy"],
        "kappa": m["cohen_kappa"],
    }

baseline_table = pd.DataFrame([result_row(k, v) for k, v in baseline_results.items()])
metric_cols = ["top1", "top2", "top3", "macro_f1", "weighted_f1", "balanced_acc", "kappa"]
display(baseline_table.sort_values("macro_f1", ascending=False).style.format({c: "{:.4f}" for c in metric_cols}))

all_grouped = baseline_results["all_reported_grouped"]["metrics"]
ppg_grouped = baseline_results["ppg_reported_grouped"]["metrics"]
degradation = pd.DataFrame([
    {"metric": k, "all_channel": all_grouped[k], "ppg_only": ppg_grouped[k], "ppg_minus_all": ppg_grouped[k] - all_grouped[k]}
    for k in ["top1_accuracy", "top2_accuracy", "top3_accuracy", "macro_f1", "balanced_accuracy", "cohen_kappa"]
])
display(degradation.style.format({"all_channel": "{:.4f}", "ppg_only": "{:.4f}", "ppg_minus_all": "{:.4f}"}))

## 12. Confusion matrices

In [ ]:
def plot_confusion_matrix(result: dict, title: str) -> None:
    cm = np.asarray(result["metrics"]["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(EMOTION_LABELS)), EMOTION_LABELS, rotation=45, ha="right")
    ax.set_yticks(range(len(EMOTION_LABELS)), EMOTION_LABELS)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(baseline_results["all_reported_grouped"], "All-channel LightGBM, grouped reported")
plot_confusion_matrix(baseline_results["ppg_reported_grouped"], "PPG-only LightGBM, grouped reported")